<a href="https://colab.research.google.com/github/adwoaadusei/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

if API_KEY is None:
    raise ValueError("GROQ_API_KEY was not found in Colab Secrets.")

print("API key loaded successfully.")

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

API key loaded successfully.
Client ready.


In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response


# Test the function
response = ask_llm("What is a microfinance institution?")

print("Answer:")
print(response.choices[0].message.content)

print("\nToken usage:")
print(response.usage)

Answer:
A microfinance institution (MFI) is an organization that provides financial services to low-income individuals, small businesses, and entrepreneurs who lack access to traditional banking services. MFIs offer a range of financial products and services, including:

1. **Microloans**: Small loans, typically ranging from $100 to $10,000, to help individuals start or expand a business, or to cover unexpected expenses.
2. **Savings accounts**: Safe and secure places for individuals to save their money, often with features like interest-bearing accounts and mobile banking.
3. **Insurance**: Life, health, and asset insurance products to help protect against risks and uncertainties.
4. **Money transfers**: Services that enable individuals to send and receive money, often across borders.
5. **Training and education**: Financial literacy programs, business skills training, and other support services to help clients manage their finances and grow their businesses.

MFIs often serve people 

1.The system role gives the model overall instructions about how it should behave and respond, while the user role contains the specific request or task. For example, a system message could say "You are a helpful financial assistant who gives clear and factual answers," while a user message could ask "What is a microfinance institution?"

2.A token is a small unit of text that an LLM processes. A token can be a whole short word, part of a longer word, punctuation, or another piece of text. API providers bill per token because the amount of text processed and generated affects the computational resources required. Charging based on tokens therefore reflects the amount of work done rather than simply charging the same amount for every API request.

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra."

print("===== TEMPERATURE 0.0 =====")

for i in range(5):
    response = ask_llm(
        question,
        temperature=0.0,
        max_tokens=100
    )
    print(f"\nRun {i+1}:")
    print(response.choices[0].message.content)


print("\n\n===== TEMPERATURE 1.2 =====")

for i in range(5):
    response = ask_llm(
        question,
        temperature=1.2,
        max_tokens=100
    )
    print(f"\nRun {i+1}:")
    print(response.choices[0].message.content)

===== TEMPERATURE 0.0 =====

Run 1:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect

Run 2:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **TradeUp Savings**: This name suggests that the savings product will help traders "trade up" and improve their financial situation.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, which is widely spoken

Run 3:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name c

At temperature 0.0, the responses were highly consistent across the five runs. The model repeatedly suggested names such as “Makola Save” and “Traders' Treasure”, with only small differences in wording. At temperature 1.2, the responses showed more variation in both the suggested names and the explanations. For example, “Market Booster” and “Sika Box” appeared at the higher temperature, while the lower temperature produced more repetitive responses. For a loan decision-support system, I would use a low temperature such as 0.0 because the system should produce consistent, factual and predictable outputs rather than highly varied responses. Higher temperatures would be more appropriate for creative tasks where generating different ideas is desirable.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

SUMMARY_PROMPT_V1 = "Summarize this loan application:\n\n{letter_text}"

print("===== L002 — V1 =====")

response_l002_v1 = ask_llm(
    SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"]),
    temperature=0.7,
    max_tokens=200
)

print(response_l002_v1.choices[0].message.content)


print("\n===== L006 — V1 =====")

response_l006_v1 = ask_llm(
    SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"]),
    temperature=0.7,
    max_tokens=200
)

print(response_l006_v1.choices[0].message.content)

SUMMARY_SYSTEM_V2 = """
You are an assistant to a microfinance loan officer.

Summarize loan application letters into a short, factual and neutral brief.
Use only information explicitly stated in the letter. Do not invent, assume,
or infer information that is not provided. Focus on the applicant, requested
loan amount, purpose of the loan, financial/business information, repayment
proposal, and collateral or guarantor information.

Write exactly 3-4 sentences. Do not make an approval or rejection decision.
"""

SUMMARY_PROMPT_V2 = """
Summarize this loan application for a busy loan officer:

{letter_text}
"""

print("===== L002 — V2 =====")

response_l002_v2 = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0,
    max_tokens=200
)

print(response_l002_v2.choices[0].message.content)


print("\n===== L006 — V2 =====")

response_l006_v2 = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0,
    max_tokens=200
)

print(response_l006_v2.choices[0].message.content)

===== L002 — V1 =====
Kwame Boateng, a commercial driver from Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his vehicle's engine and pay off personal debts. His business has been slow, but he expects it to improve after the festive season. He doesn't have collateral to offer and is relying on his future earnings to repay the loan. He's requesting urgent approval of the loan.

===== L006 — V1 =====
Kofi, a 22-year-old, is applying for a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience with these ventures and no collateral to offer, but claims to be "business-minded" and "trustworthy". He plans to repay the loan within one year, expecting his businesses to be successful by then.
===== L002 — V2 =====
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. The purpose of the loan is to repair his trotro engine and settle personal

1.What concrete problems did V1's output have that V2 fixed? V1 produced generally accurate summaries, but it sometimes made assumptions that were not directly stated in the letters. For example, V1 said that Kofi had "no prior experience" in the businesses, even though the letter only states that he had not started them yet. V2 was more careful about distinguishing stated facts from assumptions and explicitly identified missing information, such as the lack of a specific repayment schedule in L002. V2 also gave the summaries a clearer focus on information relevant to a loan officer.

2.Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature? "No invented details" is essential because a loan officer could make an unfair or incorrect decision if the system adds information that was not provided by the applicant. For example, inventing financial information, collateral, or repayment details could make an applicant appear stronger or weaker than they actually are. This failure mode is called hallucination, where an LLM generates information that is unsupported by the provided source.

In [6]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

EXTRACT_PROMPT = """
You are extracting structured information from a loan application letter.

Return ONLY a valid JSON object with EXACTLY these six keys:

{{
  "applicant_name": "string",
  "amount_ghs": "number",
  "purpose": "string",
  "monthly_profit_ghs": "number or null",
  "has_collateral_or_guarantor": "boolean",
  "repayment_months": "number or null"
}}

Rules:
1. Use only information explicitly stated in the letter.
2. Do not guess, infer, or invent information.
3. If a field is not stated in the letter, use null.
4. amount_ghs, monthly_profit_ghs, and repayment_months must be numbers, not strings.
5. Convert repayment periods stated in years into months. For example, "one year"
   means 12 months and "two years" means 24 months.
6. has_collateral_or_guarantor must be true if the letter explicitly mentions
   collateral or a guarantor, and false if the letter explicitly says there is none.
7. Return ONLY the JSON object. Do not include explanations, markdown, or
   ```json fences.

Here is one example:

LETTER:
Dear Loan Officer,

My name is Abena Owusu and I run Sweet Crumbs Bakery in Accra.
I am requesting GHS 10,000 to purchase a commercial oven and additional baking supplies.
The bakery makes approximately GHS 1,500 profit each month.
I can repay GHS 700 per month for 15 months.
My mother will act as my guarantor.

Thank you.

JSON:
{{
  "applicant_name": "Abena Owusu",
  "amount_ghs": 10000,
  "purpose": "purchase a commercial oven and additional baking supplies",
  "monthly_profit_ghs": 1500,
  "has_collateral_or_guarantor": true,
  "repayment_months": 15
}}

Now extract the information from this loan application:

{letter_text}
"""
import json

def extract_fields(letter_text):
    response = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        temperature=0.0,
        max_tokens=300
    )

    result = response.choices[0].message.content.strip()

    # Remove markdown JSON fences if the model adds them
    if result.startswith("```json"):
        result = result[7:]
    elif result.startswith("```"):
        result = result[3:]

    if result.endswith("```"):
        result = result[:-3]

    result = result.strip()

    try:
        return json.loads(result)
    except json.JSONDecodeError:
        print("Warning: Could not parse model output as JSON.")
        print("Model output:", result)
        return None


test_result = extract_fields(LETTERS["L001"])
print(json.dumps(test_result, indent=2))

import pandas as pd

extracted_results = []

for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)

    if result is not None:
        result["letter_id"] = letter_id
    else:
        result = {
            "letter_id": letter_id,
            "applicant_name": None,
            "amount_ghs": None,
            "purpose": None,
            "monthly_profit_ghs": None,
            "has_collateral_or_guarantor": None,
            "repayment_months": None
        }

    extracted_results.append(result)

extracted_df = pd.DataFrame(extracted_results)

columns = [
    "letter_id",
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

extracted_df = extracted_df[columns]

display(extracted_df)

{
  "applicant_name": "Akosua Mensah",
  "amount_ghs": 8000,
  "purpose": "buy a deep freezer and expand into frozen foods",
  "monthly_profit_ghs": 900,
  "has_collateral_or_guarantor": true,
  "repayment_months": 20
}


,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


In [7]:
l006_result = extract_fields(LETTERS["L006"])

print(json.dumps(l006_result, indent=2))

{
  "applicant_name": "Kofi",
  "amount_ghs": 50000,
  "purpose": "start a car washing business, a provision shop, and also import phones from Dubai",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": false,
  "repayment_months": 12
}


1.Why must the few-shot example NOT come from the six letters you are processing? The few-shot example should not come from the six letters being processed because it could give the model information about the actual test data and make the extraction evaluation less independent. Using a separate fictional example shows the model the required format and task without giving it the answers to the letters that will later be evaluated.

2.Why "use null, do not guess" — what did the model do without that instruction? "Use null, do not guess" is important because some information is not provided in every application. For example, L002 does not state a monthly profit or a specific repayment period, so those fields should be null rather than invented. Without this instruction, an LLM could try to fill missing information using assumptions, which could introduce inaccurate information into the system.

3.Why is temperature=0 the right choice for extraction but arguably not for creative tasks? Temperature 0 is appropriate for structured extraction because we want the model to produce consistent and predictable results when identifying information from the same letter. For creative tasks, a higher temperature can be useful because it allows more variation and different ideas to be generated.

In [8]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Prepare a concise, neutral decision-support brief using ONLY information
explicitly stated in the original loan application and the extracted JSON.

IMPORTANT:
- Do not invent, assume, speculate, or infer facts.
- Do not make judgments based on age, gender, location, writing style,
  or other personal characteristics.
- Do not treat an applicant's claims as verified facts. Clearly identify
  them as claims when relevant.
- Do not describe something as "stable", "significant", "high risk",
  "unrealistic", or similar unless the letter itself provides evidence
  supporting that characterization.
- If information is not provided, classify it as MISSING INFORMATION,
  not as a risk.
- Do not introduce generic business risks that are not directly supported
  by the application.
- Do not repeat the same point in multiple sections.
- Do not show your reasoning process or intermediate steps.

Your output MUST contain exactly these four headings and no additional
headings:

## Strengths
- List positive factors explicitly supported by the application.
- Clearly distinguish verified information from claims made by the applicant.

## Risks / Red Flags
- List concerns directly supported by information in the application,
  such as existing business difficulties, lack of collateral, or an
  unclear repayment proposal.
- Do not turn missing information into a risk.

## Missing Information
- List important information that is not provided but would help the loan
  officer assess the application.
- Do not list information that is already stated in the letter.

## Suggested Next Step
- Recommend practical actions such as requesting documents, verifying
  financial information, inviting the applicant for an interview,
  requesting a business plan, or flagging the application for senior review.
- Do NOT recommend "approve" or "reject".
- The final lending decision must always be made by a human loan officer.

Keep the entire brief concise and factual.

Original loan application:
{letter_text}

Extracted information:
{extracted_json}
"""

briefs = {}

for letter_id, letter_text in LETTERS.items():
    extracted = extracted_df[
        extracted_df["letter_id"] == letter_id
    ].iloc[0].to_dict()

    extracted_json = json.dumps(extracted, indent=2)

    response = ask_llm(
        BRIEF_PROMPT.format(
            letter_text=letter_text,
            extracted_json=extracted_json
        ),
        temperature=0.0,
        max_tokens=600
    )

    briefs[letter_id] = response.choices[0].message.content


print("===== L001 =====")
print(briefs["L001"])

print("\n===== L002 =====")
print(briefs["L002"])

print("\n===== L006 =====")
print(briefs["L006"])

print("===== L003 =====")
print(briefs["L003"])

===== L001 =====
## Strengths
- The applicant has 12 years of experience selling provisions at Makola Market.
- The applicant claims to have a monthly profit of GHS 900 from their current stall.
- The applicant has saved GHS 2,500 with the susu scheme over two years without missing a contribution.
- The applicant has a guarantor, their sister, who is a teacher.

## Risks / Red Flags
- The applicant's repayment proposal of GHS 450 monthly over 20 months may be subject to verification.
- The applicant's claim of making a monthly profit of GHS 900 has not been verified.

## Missing Information
- Detailed business plan for expanding into frozen foods.
- Verification of the applicant's monthly profit.
- Information about the applicant's sister's financial capability as a guarantor.
- Asset valuation of the deep freezer to be purchased.

## Suggested Next Step
- Request the applicant to provide a detailed business plan for the expansion into frozen foods.
- Verify the applicant's claimed mon

1. Compare the briefs for L003 and L006. Did the system identify the right strengths and red flags in each?
The system identified most of the important strengths and red flags in both applications. For L003, it correctly identified the registered business, reported monthly profit, sales records, and fixed deposit that could be pledged as collateral as strengths. It also identified useful missing information such as verification of the financial information and details about the planned expansion. However, it incorrectly described the repayment plan as potentially unrealistic even though the application did not provide enough information to support that judgment. For L006, the system correctly identified the lack of collateral, the fact that none of the proposed businesses had started, and the uncertainty of relying on future business success for repayment. Overall, the system was useful but still required human review because some of its risk assessments could be unsupported.

2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.
Practically, the LLM can make mistakes or misunderstand information, so a human loan officer should review the application and verify important information before making the final decision. Ethically, automatically approving or rejecting loans could unfairly deny applicants access to credit because of errors, biased interpretations, or differences in how applicants communicate. Keeping a human decision-maker involved provides an opportunity to identify and correct these problems.